In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta

In [ ]:
df= pd.read_csv('online_retail.csv')
display(df.head())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
#Clean the column names to be lowercase and stripped of spaces for easy coding
df.columns=df.columns.str.strip().str.lower().str.replace(' ', '_')

In [ ]:
#Drop rows where customer ID is missing
df = df.dropna(subset={'customerid'})

In [ ]:
#filter out negative quantities and price and i also had issues with creating a copy of the data frame after filtering
df=df[(df['quantity']>0)&(df['unitprice']>0)].copy()

In [ ]:
#creating a total spend column (quantity*unti price)
df['total_spend']=df['quantity']*df['unitprice']

In [ ]:
# Ensuring the invoice date is recongnised as true daetime object
df['invoice_date']=pd.to_datetime(df['invoicedate'])
print (f"\nData cleaned! Toatal rows remaining for analysis:{len(df)}")


Data cleaned! Toatal rows remaining for analysis:23626


In [ ]:
# Set our "snapshot date" to one day after the absolute last purchase in the dataset
snapshot_date = df['invoice_date'].max() + timedelta(days=1)

# Group by customer_id to calculate Recency, Frequency, and Monetary values
rfm = df.groupby('customerid').agg({
    'invoice_date': lambda x: (snapshot_date - x.max()).days, # Recency: Days since last order
    'invoiceno': 'nunique',                             # Frequency: Number of unique orders
    'total_spend': 'sum'                                      # Monetary: Total cash spent
})

# Rename the columns for clarity
rfm.rename(columns={
    'invoice_date': 'recency',
    'invoiceno': 'frequency',
    'total_spend': 'monetary'
}, inplace=True)

# Take a look at your new customer-level metrics!
print("\nFirst 5 rows of our RFM summary table:")
print(rfm.head())


First 5 rows of our RFM summary table:
            recency  frequency  monetary
customerid                              
12347.0          10          1    711.79
12348.0           1          1    892.80
12370.0           1          2   1868.02
12386.0          10          1    258.90
12395.0          14          1    346.10


In [ ]:
# 1. Score Recency: invert the scoring labels because a lower recency day count is BETTER
rfm['r_score'] = pd.qcut(rfm['recency'], q=5, labels=[5, 4, 3, 2, 1])

# 2. Score Frequency: higher frequencies are BETTER.
# (We use method='first' to handle duplicate values smoothly)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# 3. Score Monetary: higher spending is BETTER
rfm['m_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1, 2, 3, 4, 5])

# 4. Combine the scores into a single RFM text code (e.g., "555" or "111")
rfm['rfm_cell'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

print("Scoring complete! Let's view the new ranked columns:")
print(rfm[['recency', 'r_score', 'frequency', 'f_score', 'monetary', 'm_score', 'rfm_cell']].head())

Scoring complete! Let's view the new ranked columns:
            recency r_score  frequency f_score  monetary m_score rfm_cell
customerid                                                               
12347.0          10       3          1       1    711.79       5      315
12348.0           1       5          1       1    892.80       5      515
12370.0           1       5          2       4   1868.02       5      545
12386.0          10       3          1       1    258.90       2      312
12395.0          14       1          1       1    346.10       3      113


In [ ]:
# 1. Define a mapping dictionary using regular expressions (regex) to catch cell patterns
# 'r' represents Recency score, 'f' represents Frequency score
segs = {
    r'[4-5][4-5][4-5]': 'Champions',
    r'[2-5][3-5][2-5]': 'Loyal Customers',
    r'[3-5][1-2][1-5]': 'Recent Customers',
    r'[1-2][1-2][1-5]': 'Lost / Low Value',
    r'[1-2][3-5][3-5]': 'At Risk / Cant Lose Them'
}

# 2. Copy the rfm_cell text over and apply our dictionary map
rfm['segment'] = rfm['rfm_cell'].replace(segs, regex=True)

# 3. If any edge-case cells don't fall into the main buckets, fill them as 'About to Sleep'
rfm['segment'] = rfm['segment'].fillna('About to Sleep')

print("Segmentation Mapping Complete!")
# Let's see the count of customers in each group
print(rfm['segment'].value_counts())

Segmentation Mapping Complete!
segment
Loyal Customers             238
Recent Customers            197
Lost / Low Value            136
Champions                   122
At Risk / Cant Lose Them     32
331                          13
231                          13
132                          12
441                           9
142                           9
131                           8
141                           7
541                           7
431                           6
341                           5
241                           4
531                           4
152                           2
251                           2
451                           2
151                           2
551                           1
351                           1
Name: count, dtype: int64


In [ ]:
# 1. Define a clear function that checks the individual R, F, and M scores
def assign_segment(row):
    r = int(row['r_score'])
    f = int(row['f_score'])
    m = int(row['m_score'])

    # Core logical rules
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 2 and f >= 3:
        return 'Loyal Customers'
    elif r >= 3 and f <= 2:
        return 'Recent / New Customers'
    elif r <= 2 and f >= 3:
        return 'At Risk / Cant Lose Them'
    elif r <= 2 and f <= 2:
        return 'Lost / Low Value'
    else:
        return 'About to Sleep'

# 2. Apply this function across every row of your dataframe
rfm['segment'] = rfm.apply(assign_segment, axis=1)

# 3. Check the results again to ensure NO numbers are left!
print("Perfected Segmentation Mapping Breakdown:")
print(rfm['segment'].value_counts())

Perfected Segmentation Mapping Breakdown:
segment
Loyal Customers             305
Recent / New Customers      197
Lost / Low Value            136
Champions                   122
At Risk / Cant Lose Them     72
Name: count, dtype: int64


In [ ]:
# our hard work saved to a clean CSV file
rfm.to_csv("rfm_customer_segments.csv")
print("\nSuccess! 'rfm_customer_segments.csv' has been created and is ready to download.")


Success! 'rfm_customer_segments.csv' has been created and is ready to download.
